# 📡 TP Télécom 3GPP — Phase 1 (v2) : Comparaison de 3 LLMs Open Source

**Version corrigée — Métrique : Accuracy (QCM forcé A/B/C/D)**

| Modèle | Taille | Fournisseur |
|--------|--------|-------------|
| DistilGPT2 | 82M | DistilBERT / HuggingFace |
| GPT2-Small | 117M | OpenAI |
| GPT2-Medium | 345M | OpenAI |

> 💡 **Changement majeur vs v1** : on force le modèle à répondre par une lettre
> (A, B, C ou D) et on calcule l'**Accuracy** = bonnes réponses / total questions.
> C'est la métrique adaptée à un dataset QCM (recommandation du professeur).

**Pipeline global :**
```
Phase 1 → Phase 2 → Phase 3 → Phase 4 → Phase 5 → Phase 6 → Phase 7
 [Ici]    LLM+RAG  RAG Avancé  Fine-Tune   RAFT    RAG+Agent  Multi-Agent
```

## 1. Installation des dépendances

In [1]:
!pip install -q transformers torch sentencepiece
!pip install -q pandas matplotlib
print('✅ Installation terminée')

✅ Installation terminée


## 2. Imports & Configuration

In [2]:
import json, time, re
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

SAVE_DIR = r'C:\Users\HP\Documents\TP-LLM-3GPP-Pipeline'
print('✅ Imports effectués')

✅ Imports effectués


## 3. Chargement des Modèles en Local

In [3]:
from transformers import pipeline

MODELS = {
    'DistilGPT2'  : 'distilbert/distilgpt2',
    'GPT2-Small'  : 'openai-community/gpt2',
    'GPT2-Medium' : 'openai-community/gpt2-medium'
}

print('🔄 Loading models locally...')
pipelines = {}
for name, model_id in MODELS.items():
    print(f'  → Loading {name}...')
    pipelines[name] = pipeline('text-generation', model=model_id)
    print(f'  ✅ {name} ready')

print('\n✅ All models loaded and ready!')

🔄 Loading models locally...
  → Loading DistilGPT2...


Loading weights: 100%|██████████| 76/76 [00:00<00:00, 1671.31it/s]


  ✅ DistilGPT2 ready
  → Loading GPT2-Small...


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 2027.06it/s]


  ✅ GPT2-Small ready
  → Loading GPT2-Medium...


Loading weights: 100%|██████████| 292/292 [00:00<00:00, 2913.53it/s]


  ✅ GPT2-Medium ready

✅ All models loaded and ready!


## 4. Chargement du Dataset 3GPP (QCM)

In [ ]:
# -------------------------------------------------------
# 🔧 Chemins de tes fichiers dataset (TeleQnA — format QCM)
# -------------------------------------------------------
TRAIN_PATH = r'C:\Users\HP\Downloads\TeleQnA_training.txt'

with open(TRAIN_PATH, 'r', encoding='utf-8') as f:
    train_data = json.load(f)

# Conversion en paires QCM avec 4 options et lettre correcte
def index_to_letter(idx):
    return {1: 'A', 2: 'B', 3: 'C', 4: 'D'}.get(idx, None)

qa_pairs = []
for i, (key, item) in enumerate(train_data.items()):
    answer_idx = item.get('answer', None)
    correct_letter = index_to_letter(answer_idx)
    
    # ✅ Filtre : on garde uniquement les questions avec une réponse valide
    # ET les 4 options remplies
    options = [item.get(f'option {j}', '') for j in range(1, 5)]
    if correct_letter is None or any(opt == '' for opt in options):
        continue  # on ignore cette question
    
    qa_pairs.append({
        'id'             : len(qa_pairs) + 1,
        'question'       : item['question'],
        'option_A'       : options[0],
        'option_B'       : options[1],
        'option_C'       : options[2],
        'option_D'       : options[3],
        'correct_letter' : correct_letter,
        'category'       : item.get('category', '3GPP')
    })

qa_pairs_phase1 = qa_pairs[:10]
df_dataset = pd.DataFrame(qa_pairs_phase1)

print(f'✅ Dataset chargé : {len(train_data)} questions brutes')
print(f'   Questions valides (avec 4 options + réponse) : {len(qa_pairs)}')
print(f'   Utilisées pour Phase 1 : {len(qa_pairs_phase1)} questions')
print(f'\n📊 Aperçu :')
for _, row in df_dataset.head(2).iterrows():
    print(f"\n  Q{row['id']} : {row['question'][:70]}...")
    print(f"     A) {row['option_A'][:50]}")
    print(f"     B) {row['option_B'][:50]}")
    print(f"     C) {row['option_C'][:50]}")
    print(f"     D) {row['option_D'][:50]}")
    print(f"  ✅ Bonne réponse : {row['correct_letter']}")

✅ Dataset chargé : 8000 questions disponibles
   Utilisées pour Phase 1 : 10 questions

📊 Aperçu :

  Q1 : What is the purpose of the Nmfaf_3daDataManagement_Deconfigure service...
     A) To configure the MFAF to map data or analytics rec
     B) To configure the MFAF to stop mapping data or anal
     C) To supply data or analytics from the MFAF to notif
     D) To fetch data or analytics from the MFAF based on 
  ✅ Bonne réponse : A

  Q2 : Which non-orthogonal multiple access scheme utilizes the low-complexit...
     A) NOMA
     B) PDMA
     C) MUSA
     D) MUST
  ✅ Bonne réponse : D


## 5. Fonctions QCM — Prompt forcé + Extraction de la réponse

In [5]:
def build_qcm_prompt(row):
    """Construit un prompt QCM forçant une réponse A/B/C/D."""
    return (
        f"Question: {row['question']}\n"
        f"A) {row['option_A']}\n"
        f"B) {row['option_B']}\n"
        f"C) {row['option_C']}\n"
        f"D) {row['option_D']}\n"
        f"Answer (A, B, C or D):"
    )

def extract_choice(generated_text):
    """Extrait la première lettre A/B/C/D isolée trouvée dans le texte."""
    text = generated_text.strip().upper()
    match = re.search(r'\b([ABCD])\b', text)
    return match.group(1) if match else None

def call_local_model_qcm(model_name, row, max_new_tokens=5):
    """Appelle le modèle en mode QCM et retourne la lettre choisie."""
    prompt = build_qcm_prompt(row)
    t0 = time.time()
    result = pipelines[model_name](
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=50256,
        truncation=True
    )
    elapsed = time.time() - t0
    full_text = result[0]['generated_text']
    generated = full_text.replace(prompt, '').strip()
    choice = extract_choice(generated)
    return choice, generated, elapsed

print('✅ Fonctions QCM définies')

# Test rapide sur la première question
test_choice, test_raw, test_time = call_local_model_qcm('DistilGPT2', df_dataset.iloc[0])
print(f'\n🔍 Test : choix détecté = {test_choice} | brut = "{test_raw[:30]}" | {test_time:.2f}s')

[transformers] Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=5) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ Fonctions QCM définies


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



🔍 Test : choix détecté = None | brut = "The MFAF is" | 3.03s


## 6. Inférence QCM sur les 3 Modèles

In [6]:
results = []

for model_name in MODELS.keys():
    print(f'\n{"="*60}')
    print(f'🤖 MODEL : {model_name}')
    print('='*60)

    for _, row in df_dataset.iterrows():
        print(f'  → Question {row["id"]} : {row["question"][:55]}...')
        choice, raw, elapsed = call_local_model_qcm(model_name, row)
        is_correct = (choice == row['correct_letter'])

        results.append({
            'model'           : model_name,
            'question_id'     : row['id'],
            'question'        : row['question'],
            'correct_letter'  : row['correct_letter'],
            'predicted_letter': choice,
            'raw_output'      : raw,
            'is_correct'      : is_correct,
            'inference_time_s': round(elapsed, 2)
        })
        status = '✅' if is_correct else '❌'
        print(f'     {status} Prédit: {choice} | Correct: {row["correct_letter"]} | {elapsed:.2f}s')

df_results = pd.DataFrame(results)
df_results.to_csv(f'{SAVE_DIR}\\Phase1\\phase1_raw_results_v2.csv', index=False)
print(f'\n✅ Done — {len(df_results)} résultats sauvegardés')

[transformers] Both `max_new_tokens` (=5) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🤖 MODEL : DistilGPT2
  → Question 1 : What is the purpose of the Nmfaf_3daDataManagement_Deco...


[transformers] Both `max_new_tokens` (=5) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


     ❌ Prédit: None | Correct: A | 0.96s
  → Question 2 : Which non-orthogonal multiple access scheme utilizes th...


[transformers] Both `max_new_tokens` (=5) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

     ✅ Prédit: A | Correct: A | 3.23s
  → Question 8 : What is one of the use cases of SDN (Software Defined N...


[transformers] Both `max_new_tokens` (=5) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


     ❌ Prédit: A | Correct: nan | 3.42s
  → Question 9 : What is the primary concern of proposals aiming to impr...


[transformers] Both `max_new_tokens` (=5) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


     ❌ Prédit: A | Correct: nan | 3.58s
  → Question 10 : What does the acronym VPN stand for?...
     ✅ Prédit: A | Correct: A | 2.92s

✅ Done — 30 résultats sauvegardés


## 7. Calcul de l'Accuracy par Modèle

In [ ]:
eval_results = []

for model_name in MODELS.keys():
    subset = df_results[df_results['model'] == model_name]
    correct = subset['is_correct'].sum()
    total   = len(subset)
    accuracy = correct / total if total > 0 else 0
    avg_time = subset['inference_time_s'].mean()

    eval_results.append({
        'Modèle'        : model_name,
        'Bonnes réponses' : f'{correct}/{total}',
        'Accuracy'      : round(accuracy, 4),
        'Temps moy. (s)': round(avg_time, 2)
    })

df_eval = pd.DataFrame(eval_results)
print('📊 Tableau Accuracy — Phase 1 :')
print(df_eval.to_string(index=False))

df_eval.to_csv(f'{SAVE_DIR}\\Phase1\\phase1_evaluation_v2.csv', index=False)
print('\n💾 Sauvegardé → Phase1/phase1_evaluation_v2.csv')

## 8. Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Comparaison des LLMs — Accuracy (QCM) sur Dataset 3GPP', fontsize=15, fontweight='bold')
palette = ['#2196F3', '#FF5722', '#4CAF50']

# --- Accuracy ---
ax1 = axes[0]
bars = ax1.bar(df_eval['Modèle'], df_eval['Accuracy']*100, color=palette, alpha=0.85)
for b in bars:
    ax1.text(b.get_x()+b.get_width()/2, b.get_height()+1,
             f'{b.get_height():.0f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax1.set_ylim(0, 100)
ax1.set_title('Accuracy (%)', fontweight='bold')
ax1.set_ylabel('Accuracy (%)')
ax1.grid(axis='y', alpha=0.3)

# --- Temps ---
ax2 = axes[1]
bars = ax2.bar(df_eval['Modèle'], df_eval['Temps moy. (s)'], color=palette, alpha=0.85)
for b in bars:
    ax2.text(b.get_x()+b.get_width()/2, b.get_height()+0.1,
             f'{b.get_height():.1f}s', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax2.set_title("Temps moyen d'inférence", fontweight='bold')
ax2.set_ylabel('Secondes')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}\\Phase1\\phase1_comparaison_llms_v2.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Graphique → Phase1/phase1_comparaison_llms_v2.png')

## 9. Sélection du Meilleur Modèle

In [ ]:
# Score composite : Accuracy (75%) + Vitesse (25%)
df_eval['Score_composite'] = (
    0.75 * df_eval['Accuracy'] +
    0.25 * (1 / df_eval['Temps moy. (s)'] /
            (1 / df_eval['Temps moy. (s)']).max())
).round(4)

df_sorted  = df_eval.sort_values('Score_composite', ascending=False)
best_model = df_sorted.iloc[0]['Modèle']
best_model_id = MODELS[best_model]

print('📊 Classement final :')
print(df_sorted[['Modèle','Accuracy','Temps moy. (s)','Score_composite']].to_string(index=False))
print(f'\n🏆 Meilleur modèle → {best_model}')

config = {
    'best_model_name' : best_model,
    'best_model_id'   : best_model_id,
    'phase1_complete' : True,
    'metric_used'     : 'accuracy'
}
with open(f'{SAVE_DIR}\\pipeline_config_v2.json', 'w') as f:
    json.dump(config, f, indent=2)

print('💾 Config sauvegardée → pipeline_config_v2.json')
print('\n➡️  Phase 1 (v2) terminée ! Passez au notebook Phase 2 — LLM + RAG')

---
## ✅ Phase 1 (v2) Terminée

**Changement clé vs v1 :**
- Métrique = **Accuracy** (bonnes réponses / total) au lieu de ROUGE
- Format = **QCM forcé A/B/C/D** au lieu de génération libre

**Fichiers produits :**
- `Phase1/phase1_raw_results_v2.csv`
- `Phase1/phase1_evaluation_v2.csv`
- `Phase1/phase1_comparaison_llms_v2.png`
- `pipeline_config_v2.json`

**➡️ Prochaine étape : Phase 2 (v2) — LLM + RAG avec TSpec-LLM**